In [1]:
import sys
print(sys.version)

3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]


# AdvTG — end-to-end pipeline

Adversarial HTTP traffic generation vs. DL malicious-traffic detectors, run stage by stage:

1. **Dataset** → 2. **Detectors** (token + image) → 3. **LLM finetune** (Unsloth) → 4. **PPO** adversarial generation

**Where this runs:** the kernel is a **Colab GPU VM** — whether you open this on colab.research.google.com or connect VS Code to a Colab runtime, the cells execute on Colab's VM. So:

- The VM only has what's in the **git remote**. **Push your fork first**; the Setup cell clones it (and `git pull`s on re-runs). Your local VS Code edits do **not** sync to the VM.
- Give the runtime a GPU: Colab ▸ Runtime ▸ Change runtime type ▸ **T4**.

Stages 3 and 4 pin **different `trl` versions**, so **restart the kernel** between them — VS Code: the **↻ Restart** button in the notebook toolbar · Colab web: Runtime ▸ Restart — then re-run **Setup + Config**. Each stage flags this.

In [3]:
import os, sys, subprocess

# Runs on the Colab VM. The VM sees only the git remote, so push your branch first;
# this clones it, and pulls the latest on any re-run.
GIT_URL = "https://github.com/TejaswiMN/AdvTG.git"
BRANCH  = "punyam"

if os.path.isfile("gen_synthetic_data.py"):
    REPO = os.getcwd()                                    # already inside the repo
else:
    REPO = "/content/AdvTG"
    if os.path.isdir(os.path.join(REPO, ".git")):
        subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "-b", BRANCH, GIT_URL, REPO], check=True)

os.chdir(REPO)
sys.path.insert(0, REPO)                                  # make the DL package importable
print("repo:", REPO)
print("has gen_synthetic_data.py:", os.path.isfile("gen_synthetic_data.py"))

repo: /content/AdvTG
has gen_synthetic_data.py: True


## Config — set everything here

Switch datasets by changing `DATASET_SOURCE`. Everything downstream reads `dataset/train_data2.json`.

In [4]:
import os

# ─── the one knob: where dataset/train_data2.json comes from ──────────────────
DATASET_SOURCE = "synthetic"      # "synthetic" | "cicids2017"

# synthetic sizes
N_TRAIN         = 20000
N_TEST          = 4000
MALICIOUS_RATIO = 0.35

# CIC-IDS2017 (only used when DATASET_SOURCE == "cicids2017")
KAGGLE_DATASET       = "chethuhn/network-intrusion-dataset"
CICIDS_MAX_PER_CLASS = 20000

# detector training
MAX_LENGTH = 512
BATCH_SIZE = 16
NUM_EPOCHS = 2
TRAIN_BERT = False                # heavy, and not used by the PPO reward

# paths (match the repo's ../dataset and ../model conventions)
DATA_DIR   = os.path.join(REPO, "dataset")
MODEL_DIR  = os.path.join(REPO, "model")
TRAIN_JSON = os.path.join(DATA_DIR, "train_data2.json")
TEST_JSON  = os.path.join(DATA_DIR, "test2.json")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print("dataset source:", DATASET_SOURCE)

dataset source: synthetic


## Stage 1 — Dataset

Produces `dataset/train_data2.json` (+ `test2.json`) — the single file every downstream stage reads.

In [4]:
import shutil

if DATASET_SOURCE == "synthetic":
    !python gen_synthetic_data.py --n {N_TRAIN} --test-n {N_TEST} \
        --malicious-ratio {MALICIOUS_RATIO} --out "{TRAIN_JSON}" --test-out "{TEST_JSON}"

elif DATASET_SOURCE == "cicids2017":
    # needs a Kaggle token at ~/.kaggle/kaggle.json (Kaggle ▸ Settings ▸ API ▸ Create New Token)
    !pip -q install kaggle
    !apt-get -qq install -y tshark >/dev/null
    !kaggle datasets download -d {KAGGLE_DATASET} -p /content/cicids --unzip
    # PCAPs carry the HTTP text; TrafficLabelling CSVs carry the labels (join on the 5-tuple).
    !python build_train_data.py \
        --pcap-dir /content/cicids/PCAPs \
        --csv-dir  /content/cicids/TrafficLabelling \
        --max-per-class {CICIDS_MAX_PER_CLASS} \
        --out "{TRAIN_JSON}"
    shutil.copy(TRAIN_JSON, TEST_JSON)   # PPO reads a held-out file; reuse the extract

else:
    raise ValueError("DATASET_SOURCE must be 'synthetic' or 'cicids2017'")

wrote 20000 records to /content/AdvTG/dataset/train_data2.json  {'Benign': 13000, 'Malicious': 7000}
wrote 4000 records to /content/AdvTG/dataset/test2.json


In [5]:
import json, collections
recs = json.load(open(TRAIN_JSON, encoding="utf-8"))
print(len(recs), "records", dict(collections.Counter(r["Label"] for r in recs)))
r = recs[0]
print("\n" + r["Request Line"])
for k, v in list(r["Request Headers"].items())[:4]:
    print(f"  {k}: {v}")
print("  ->", r["Label"], "| source:", r["Source"])

20000 records {'Benign': 13000, 'Malicious': 7000}

GET /blog/2024/05/release-notes HTTP/1.1
  Host: shop.example.com
  User-Agent: Mozilla/5.0 (X11; Linux x86_64; rv:125.0) Gecko/20100101 Firefox/125.0
  Accept: text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8
  Accept-Language: es-ES,es;q=0.9
  -> Benign | source: synthetic-browse


## Stage 2 — Detector training

Trains the token-level (TextCNN / CNN-LSTM / DNN) and image detectors, then writes the `model_configs` pickles the PPO stage attacks. Runs on CPU or GPU.

In [6]:
!pip -q install "transformers==4.44.2" "datasets==2.21.0" "accelerate>=0.33.0" "scikit-learn>=1.3.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 63.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 109.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 67.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 46.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.

In [7]:
# token-level detectors (TextCNN / CNN-LSTM / DNN), sharing the BERT tokenizer's vocab
import os, torch
import DL.training as _T
from transformers import TrainingArguments, AutoTokenizer
from DL.data_processing import load_data, prepare_dataset
from DL.models import TextCNNClassifier, CNNLSTMClassifier, DNNClassifier
from DL.training import train_custom_model, train_transformer_model

_T.MODEL_PATH = MODEL_DIR          # repo hardcodes ./models/; redirect saves under model/
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

TOKENIZER_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
data = load_data(TRAIN_JSON)
train_ds, val_ds, test_ds = prepare_dataset(data, tokenizer, MAX_LENGTH)

vocab_size, embed_size, num_classes = len(tokenizer.vocab), 128, 2
os.makedirs(os.path.join(MODEL_DIR, "custom_models"), exist_ok=True)
args = TrainingArguments(output_dir=os.path.join(MODEL_DIR, "custom_models"),
                         per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
                         learning_rate=2e-5, num_train_epochs=NUM_EPOCHS, report_to="none")

for name, model in {
        "textcnn":  TextCNNClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH),
        "cnn_lstm": CNNLSTMClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH),
        "dnn":      DNNClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH)}.items():
    print("training", name)
    train_custom_model(model, name, train_ds, val_ds, args)

if TRAIN_BERT:
    bargs = TrainingArguments(output_dir=os.path.join(MODEL_DIR, "bert"),
                              evaluation_strategy="epoch", learning_rate=2e-5,
                              per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
                              num_train_epochs=NUM_EPOCHS, weight_decay=0.01, save_strategy="epoch",
                              load_best_model_at_end=True, report_to="none")
    train_transformer_model("bert", TOKENIZER_NAME, train_ds, val_ds, bargs)

device: cuda


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

training textcnn
Epoch 1, Eval Loss: 27.019442826509476, Accuracy: 0.913125, Precision: 0.9233759550084889, Recall: 0.913125, F1: 0.9097061852346912
Epoch 2, Eval Loss: 13.160168495029211, Accuracy: 0.9825, Precision: 0.9826908646772057, Recall: 0.9825, F1: 0.9824246105962121
training cnn_lstm
Epoch 1, Eval Loss: 3.258777469396591, Accuracy: 0.9975, Precision: 0.9975095877277086, Recall: 0.9975, F1: 0.9974979292650695
Epoch 2, Eval Loss: 0.5043738714884967, Accuracy: 1.0, Precision: 1.0, Recall: 1.0, F1: 1.0
training dnn
Epoch 1, Eval Loss: 10.037576308473945, Accuracy: 0.97375, Precision: 0.9739642665692648, Recall: 0.97375, F1: 0.9736129720328804
Epoch 2, Eval Loss: 4.580714910989627, Accuracy: 0.9875, Precision: 0.9877360717658168, Recall: 0.9875, F1: 0.9874461504258657


In [10]:
# image-based detectors: each request rendered as a 28x28 byte image (ord(c) % 128)
import numpy as np, torch, os
from torch.utils.data import DataLoader, TensorDataset
from DL.data_processing import json_to_string
from DL.image_models import ImageCNN, ImageMLP

IMG = (28, 28)
def to_image(it):
    text = it["Request Line"] + "\n" + json_to_string(it["Request Headers"]) + "\n\n" + it["Request Body"]
    v = [ord(c) % 128 for c in text][:IMG[0] * IMG[1]]
    v += [0] * (IMG[0] * IMG[1] - len(v))
    return np.array(v, dtype=np.float32).reshape(IMG)

X = torch.tensor(np.stack([to_image(r) for r in data]))
y = torch.tensor([1 if r["Label"] == "Malicious" else 0 for r in data], dtype=torch.long)
k = int(len(X) * 0.9)
loader = DataLoader(TensorDataset(X[:k], y[:k]), batch_size=64, shuffle=True)

for name, m in {"imagecnn": ImageCNN(), "imagemlp": ImageMLP()}.items():
    m.to(device); opt = torch.optim.Adam(m.parameters(), 1e-3); lf = torch.nn.CrossEntropyLoss()
    for _ in range(int(NUM_EPOCHS)):
        m.train()
        for xb, yb in loader:
            loss = lf(m(xb.to(device)), yb.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
    m.eval()
    with torch.no_grad():
        acc = (m(X[k:].to(device)).argmax(1).cpu() == y[k:]).float().mean().item()
    path = os.path.join(MODEL_DIR, "custom_models", name + ".bin")
    torch.save(m.state_dict(), path)
    print(f"{name}: val acc {acc:.3f} -> {path}")

imagecnn: val acc 0.947 -> /content/AdvTG/model/custom_models/imagecnn.bin
imagemlp: val acc 0.905 -> /content/AdvTG/model/custom_models/imagemlp.bin


In [11]:
import pickle, os
from transformers import AutoTokenizer
from DL.models import TextCNNClassifier, CNNLSTMClassifier, DNNClassifier
from DL.image_models import ImageCNN, ImageMLP

cm = os.path.join(MODEL_DIR, "custom_models")

def cfg(name, cls):
    return {"type": "custom", "name": name, "path": os.path.join(cm, name + ".bin"), "class": cls}

text_configs = [
    cfg("textcnn",  TextCNNClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH)),
    cfg("cnn_lstm", CNNLSTMClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH)),
    cfg("dnn",      DNNClassifier(vocab_size, embed_size, num_classes, MAX_LENGTH)),
]
image_configs = [cfg("imagecnn", ImageCNN()), cfg("imagemlp", ImageMLP())]

pickle.dump(text_configs,  open(os.path.join(MODEL_DIR, "model_configs.pkl"), "wb"))
pickle.dump(image_configs, open(os.path.join(MODEL_DIR, "imgae_model_configs.pkl"), "wb"))

# PPO's Text reward re-tokenises responses with this — save it even when BERT is skipped.
AutoTokenizer.from_pretrained(TOKENIZER_NAME).save_pretrained(os.path.join(MODEL_DIR, "bert"))
print("wrote model_configs.pkl, imgae_model_configs.pkl, model/bert/ tokenizer")

wrote model_configs.pkl, imgae_model_configs.pkl, model/bert/ tokenizer


## Stage 3 — LLM finetuning (Unsloth, GPU)

LoRA SFT of Llama-3-8b to generate benign/malicious traffic in the dataset's format. Fits a T4/L4.

> **Requires Python 3.11** (top cell). On Colab's default 3.13 the pinned stack won't install.
>
> **Restart the kernel** first (coming from Stage 2's different `trl`) — Colab: Runtime ▸ Restart · VS Code: **↻ Restart** — then re-run **Setup + Config**, then the cells below **in order**.
>
> The install cell pins **torch 2.3.1 + xformers 0.0.26.post1**, because `unsloth 2024.8` needs torch ≤ 2.3 but Colab ships torch 2.6. Run **install → training** in order and no extra restart is needed (torch is first imported in the training cell). torch 2.3.1 then persists into Stage 4.

In [ ]:
# Unsloth stack for Python 3.11. unsloth 2024.8 needs trl<0.9.0 (-> trl 0.8.6, which still
# has the SFTTrainer(...) kwargs this notebook uses), torch<=2.3 + xformers<0.0.27, and
# bitsandbytes<0.44 (newer bnb needs torch>=2.4). accelerate is pinned ==0.33.0: newer
# accelerate's dispatch_model calls model.to() on the 4-bit model, which transformers 4.44.2
# forbids ("`.to` is not supported for 4-bit"). Colab's 3.11 ships torch 2.6, so install
# torch 2.3.1 WITH its matched CUDA runtime (cuDNN 8 + triton 2.3.1).
!pip -q install "unsloth==2024.8" "trl==0.8.6" "transformers==4.44.2" "datasets==2.21.0" \
    "peft>=0.12.0" "accelerate==0.33.0" "bitsandbytes==0.43.1"
!pip -q install "torch==2.3.1" "torchvision==0.18.1"
!pip -q install --no-deps "xformers==0.0.26.post1"
print(">> torch 2.3.1 + xformers 0.0.26.post1 + trl 0.8.6 + bnb 0.43.1 + accelerate 0.33.0 — run training next")

In [3]:
!pip -q install "bitsandbytes==0.43.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 11.6 MB/s eta 0:00:0000:0100:01


In [1]:
!pip -q install --force-reinstall --no-deps "trl==0.8.6"

In [4]:
!pip -q install --force-reinstall --no-deps "accelerate==0.33.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 3.2 MB/s eta 0:00:00a 0:00:01


In [3]:
import os, json, torch
assert torch.cuda.is_available(), "Stage 3 needs a GPU runtime"

from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

MAX_SEQ = 2048
model, tok = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit", max_seq_length=MAX_SEQ,
    dtype=None, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407)

def json_to_string(d, indent=0):
    out, pad = [], " " * indent
    if isinstance(d, dict):
        for k, v in d.items():
            if isinstance(v, (dict, list)):
                out.append(f"{pad}{k}:"); out.append(json_to_string(v, indent + 1))
            else:
                out.append(f"{pad}{k}: {v}")
    elif isinstance(d, list):
        for it in d:
            out.append(json_to_string(it, indent))
    else:
        out.append(f"{pad}{d}")
    return "\n".join(out)

alpaca = ("Below is an instruction that describes a task, paired with an input that provides "
          "further context. Write a response that appropriately completes the request.\n\n"
          "### Instruction:\n{}\n\n### Input:\n{}\n\n### Response:\n{}")
EOS = tok.eos_token
data = json.load(open(TRAIN_JSON, encoding="utf-8"))
def body(it): return it["Request Line"] + "\n" + json_to_string(it["Request Headers"]) + "\n\n" + it["Request Body"]
texts = [alpaca.format(
            "Follow these tips to generate malicious http traffic" if it["Label"] == "Malicious"
            else "Follow these tips to generate benign http traffic",
            it["Request Line"], body(it)) + EOS
         for it in data]
ds = Dataset.from_dict({"text": texts}).shuffle(seed=42)

trainer = SFTTrainer(
    model=model, tokenizer=tok, train_dataset=ds,
    dataset_text_field="text", max_seq_length=MAX_SEQ, packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        warmup_steps=5, max_steps=60, learning_rate=2e-4,
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=5, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=3407,
        output_dir=os.path.join(MODEL_DIR, "llama_outputs"), report_to="none"))
trainer.train()
model.save_pretrained(os.path.join(MODEL_DIR, "llama_lora"))
tok.save_pretrained(os.path.join(MODEL_DIR, "llama_lora"))
print("saved LoRA ->", os.path.join(MODEL_DIR, "llama_lora"))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
==((====))==  Unsloth 2024.8: Fast Llama patching. Transformers = 4.44.2.
   \\   /|    GPU: Tesla T4. Max memory: 14.563 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.3.1+cu121. CUDA = 7.5. CUDA Toolkit = 12.1.
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.26.post1. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth 2024.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

max_steps is given, it will override any value given in num_train_epochs
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 20,000 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 8
\        /    Total batch size = 16 | Total steps = 60
 "-____-"     Number of trainable parameters = 41,943,040


Step,Training Loss
5,2.125300
10,1.395300
15,1.079800
20,0.925100
25,0.822500
30,0.858900
35,0.883500
40,0.807500
45,0.885400
50,0.893700


saved LoRA -> /content/AdvTG/model/llama_lora


## Stage 4 — PPO adversarial generation (GPU)

PPO tunes a generator (`EleutherAI/pythia-160m`) to flip the frozen detectors' predictions. Reward = detector score for the *opposite* label. `FEATURE_TYPE` picks which detector family to attack.

> Uses the **classic `trl==0.8.6`** PPO API, which conflicts with Stage 3. **Restart the kernel** first — VS Code: **↻ Restart** · Colab: Runtime ▸ Restart — then re-run **Setup + Config**, then the cells below.

In [3]:
# classic trl PPO API for Python 3.11 (conflicts with Stage 3's trl — restart first).
# Same guardrails as Stage 3: torch 2.3.1 WITH its CUDA runtime (cuDNN 8), bitsandbytes 0.43.1
# (newer bnb needs torch>=2.4), and accelerate==0.33.0 (open pins pull a too-new accelerate).
# The PPO policy here is fp16 pythia-160m (not 4-bit), so this stage is lighter than Stage 3.
!pip -q install "trl==0.8.6" "transformers==4.44.2" "peft>=0.11.0" "accelerate==0.33.0" "bitsandbytes==0.43.1"
!pip -q install "torch==2.3.1" "torchvision==0.18.1"
print(">> torch 2.3.1 (+cuDNN8) + bnb 0.43.1 + accelerate 0.33.0 — run the PPO cell next")

>> torch 2.3.1 (+cuDNN8) + bnb 0.43.1 + accelerate 0.33.0 — run the PPO cell next


In [5]:
import os, sys, pickle, torch
import torch.nn.functional as F

RL_DIR = os.path.join(REPO, "RL-Adv")
sys.path.insert(0, RL_DIR)
os.chdir(RL_DIR)   # so the repo's ../model and ../dataset paths resolve

from config import (create_ppo_config, features_dict, device, generation_kwargs,
                    columns_to_log, output_min_length, output_max_length,
                    max_length, query_max_length)
from data_utils import load_http_dataset, create_dataloader, text2image
from model_utils import setup_models, prepare_query_tensors, evaluate_responses
from utils import set_seed, save_results, mkdir
from trl import PPOTrainer
from trl.core import LengthSampler

FEATURE_TYPE  = "Text"    # "Text" | "Image" — which detector family to attack
MAX_PPO_STEPS = 40
SAMPLE_SIZE   = 2000

set_seed(42)
config = create_ppo_config()
config.is_peft_model = False   # PPO a plain fp16 policy, not a LoRA adapter
config.log_with = None         # skip wandb

dataset    = load_http_dataset(file_path="../dataset/test2.json", sample_size=SAMPLE_SIZE)
dataloader = create_dataloader(dataset, batch_size=config.batch_size)

ppo_model, ref_model, tokenizer = setup_models(config.model_name, device, load_in_4bit=False)
generation_kwargs["pad_token_id"] = tokenizer.eos_token_id
generation_kwargs["top_k"] = 0
ppo_trainer = PPOTrainer(config, ppo_model, ref_model, tokenizer, dataset=dataset)

model_configs = pickle.load(open(features_dict[FEATURE_TYPE], "rb"))  # skips the input() prompt
out_sampler   = LengthSampler(output_min_length, output_max_length)

test_tokenizer = None
if FEATURE_TYPE == "Text":
    from transformers import AutoTokenizer
    test_tokenizer = AutoTokenizer.from_pretrained("../model/bert/")

all_data = []
for step, batch in enumerate(dataloader):
    if step >= MAX_PPO_STEPS:
        break
    query_tensors, origin_label, requirement_label, _ = prepare_query_tensors(
        batch, tokenizer, device, query_max_length)
    generation_kwargs["max_new_tokens"] = out_sampler()
    resp = ppo_trainer.generate(query_tensors, **generation_kwargs)
    response_tensors = [r.squeeze()[:max_length] for r in resp]
    batch["response"] = [tokenizer.decode(r.squeeze()) for r in response_tensors]

    if FEATURE_TYPE == "Text":
        texts  = [r.split("\n", 1)[-1][:max_length] for r in batch["response"]]
        tt     = [torch.tensor(test_tokenizer(t)["input_ids"]).to(device) for t in texts]
        padded = [F.pad(t, (0, max_length - t.size(0))) if t.size(0) < max_length else t[:max_length] for t in tt]
        feats  = torch.stack(padded)
    else:
        feats  = torch.stack(text2image(batch["response"])).to(device)

    rewards, pred = evaluate_responses(batch, FEATURE_TYPE, model_configs, feats, device, requirement_label)
    stats = ppo_trainer.step(query_tensors, response_tensors, list(rewards))
    ppo_trainer.log_stats(stats, batch, rewards, columns_to_log=columns_to_log)

    hit = (pred.cpu() == torch.tensor(requirement_label)).float().mean().item()
    print(f"step {step:02d}  reward {rewards.mean():.3f}  target-hit {hit:.2f}")

    for i in range(len(batch["instruction"])):
        all_data.append({"Request Line": batch["input"][i], "Label": origin_label[i],
                         "Origin Output": batch["output"][i], "Request Body": "",
                         "Request Headers": batch["response"][i]})

save_results(all_data, FEATURE_TYPE, 0)
save_path = os.path.join("../model/ppo_model", FEATURE_TYPE)
mkdir(save_path)
ppo_model.save_pretrained(save_path); tokenizer.save_pretrained(save_path)
os.chdir(REPO)
print("saved PPO policy ->", os.path.join(MODEL_DIR, "ppo_model", FEATURE_TYPE))

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
You're using a GPTNeoXTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a

step 00  reward -0.180  target-hit 0.50
step 01  reward 0.860  target-hit 0.75


/usr/local/lib/python3.11/dist-packages/trl/trainer/ppo_trainer.py:1222: UserWarning: The average ratio of batch (13.95) exceeds threshold 10.00. Skipping batch.
  warnings.warn(


step 02  reward 0.651  target-hit 0.75
step 03  reward 0.437  target-hit 0.75
step 04  reward 0.652  target-hit 0.75


/usr/local/lib/python3.11/dist-packages/trl/trainer/ppo_trainer.py:1222: UserWarning: The average ratio of batch (54.17) exceeds threshold 10.00. Skipping batch.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/trl/trainer/ppo_trainer.py:1222: UserWarning: The average ratio of batch (278.61) exceeds threshold 10.00. Skipping batch.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/trl/trainer/ppo_trainer.py:1222: UserWarning: The average ratio of batch (540.01) exceeds threshold 10.00. Skipping batch.
  warnings.warn(


step 05  reward 0.663  target-hit 0.50
step 06  reward 0.495  target-hit 0.50
step 07  reward 0.231  target-hit 0.75
step 08  reward 0.289  target-hit 0.50
step 09  reward -0.669  target-hit 0.00
step 10  reward 0.106  target-hit 0.25
step 11  reward 0.167  target-hit 0.50
step 12  reward 0.085  target-hit 0.25
step 13  reward 0.334  target-hit 0.75
step 14  reward -0.468  target-hit 0.00
step 15  reward 0.291  target-hit 0.75
step 16  reward 0.416  target-hit 0.75
step 17  reward -0.009  target-hit 0.50
step 18  reward -0.042  target-hit 0.25
step 19  reward 0.573  target-hit 0.75
step 20  reward -0.157  target-hit 0.25
step 21  reward 0.027  target-hit 0.25
step 22  reward -0.331  target-hit 0.25
step 23  reward 0.357  target-hit 0.50
step 24  reward -0.241  target-hit 0.25
step 25  reward 0.163  target-hit 0.25
step 26  reward 0.301  target-hit 0.75
step 27  reward 0.048  target-hit 0.50
step 28  reward 0.035  target-hit 0.25
step 29  reward 0.338  target-hit 0.50
step 30  reward 0.

/usr/local/lib/python3.11/dist-packages/trl/trainer/ppo_trainer.py:1222: UserWarning: The average ratio of batch (14.42) exceeds threshold 10.00. Skipping batch.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/trl/trainer/ppo_trainer.py:1222: UserWarning: The average ratio of batch (34.61) exceeds threshold 10.00. Skipping batch.
  warnings.warn(


step 38  reward -0.184  target-hit 0.50
step 39  reward 0.013  target-hit 0.25
saved PPO policy -> /content/AdvTG/model/ppo_model/Text


In [6]:
import glob, json, os

files = sorted(glob.glob(os.path.join(REPO, "dataset", "PPO_data", "**", "*.json"), recursive=True))
if files:
    print("latest:", files[-1])
    print(json.dumps(json.load(open(files[-1]))[:2], indent=2)[:1500])
else:
    print("no PPO_data yet — run Stage 4 first")

latest: /content/AdvTG/dataset/PPO_data/Text/new333/0.json
[
  {
    "Request Line": "POST /blog/comment HTTP/1.1",
    "Label": 1,
    "Origin Output": "POST /blog/comment HTTP/1.1\nHost: portal.example.net\nUser-Agent: DirBuster-1.0-RC1 (http://www.owasp.org/index.php/Category:OWASP_DirBuster_Project)\nAccept: */*\nAccept-Language: en-US,en;q=0.9\nAccept-Encoding: gzip, deflate\nContent-Type: application/x-www-form-urlencoded\nContent-Length: 56\nX-Requested-With: XMLHttpRequest\n\nname=frank&comment=<script>alert(1)</script>&submit=Post",
    "Request Body": "",
    "Request Headers": "Follow these tips to generate benign http traffic \nGET advertising/search engines in bricklayers/traders/business etc., then get traffic bulletins and statistics concerning your publishing, direct to your editor or authors\nhttps://www.google.com/dp/googlepages/affilirxtintro?41&26=... through Google Manage Your (popular) article creation, posting logos, documentation and workspaces in professionals.